In [2]:
import sys
sys.path.insert(0, '/root/autodl-tmp')

from Strategy.label.label_generator import LabelGenerator

lg = LabelGenerator(time_start=1430, time_end=1457, price_type="twap")
price_path, label_path = lg.generate_and_save()
print(f"价格表: {price_path}")
print(f"Label:  {label_path}")

LabelGenerator [TWAP_1430_1457]:   0%|          | 0/1261 [00:00<?, ?day/s]

价格表: /root/autodl-tmp/Strategy/outputs/labels/TWAP_1430_1457.fea
Label:  /root/autodl-tmp/Strategy/outputs/labels/LABEL_TWAP_1430_1457.fea


In [1]:
import sys
sys.path.insert(0, '/root/autodl-tmp')

from Strategy.label.label_generator import generate_and_save_open0935_1000_label

# 全市场分钟数据范围内生成并落盘 (与 TWAP Label 同目录 outputs/labels/)
# 可选: start_date=20210104, end_date=20251231 缩小范围以加快调试
# save_price_tables=True 会额外保存 09:35/10:00 两张 open 宽表, 便于回测直接读价
open_label_path = generate_and_save_open0935_1000_label(save_price_tables=True)
print(f"OPEN0935->次日10:00 Label: {open_label_path}")


Error.  nthreads must be a positive integer

Open@935:   0%|          | 0/1261 [00:00<?, ?day/s]

Open@1000:   0%|          | 0/1261 [00:00<?, ?day/s]

OPEN0935->次日10:00 Label: /root/autodl-tmp/Strategy/outputs/labels/LABEL_OPEN0935_1000.fea


In [ ]:
import sys, logging
sys.path.insert(0, '/root/autodl-tmp')
logging.basicConfig(level=logging.INFO)

from Strategy import config
from Strategy.factor.daily_factor_library import DailyFactorLibraryAdapter

# 日频因子需覆盖 训练+验证+样本外 至数据末，否则 val / oos / 打分缺特征（见 config 注释）
# end_date 不传 = 用 Daily_data 中最后交易日；仅调试可写 end_date=config.TRAIN_END
adapter = DailyFactorLibraryAdapter()
saved = adapter.compute_and_save_all(
    start_date=config.TRAIN_START,
    # end_date=config.TRAIN_END,  # 仅训练内调试时取消注释
)
print(f"已保存 {len(saved)} 个因子")

In [ ]:
import sys, logging
sys.path.insert(0, '/root/autodl-tmp')
logging.basicConfig(level=logging.INFO)

from Strategy.factor.factor_base import FactorRegistry
import Strategy.factor.minute_derived_factors   # 触发注册
import Strategy.factor.custom_factors           # 触发注册
 
FactorRegistry.compute_all()

In [ ]:
import sys, logging
sys.path.insert(0, '/root/autodl-tmp')
logging.basicConfig(level=logging.INFO)

from Strategy.factor.minute_derived_factors import JumpVariationFactor

jv = JumpVariationFactor()
jv.compute_and_save()  # 输出 RV / RVC / RVJ 等 13 个因子

In [5]:
import sys
sys.path.insert(0, '/root/autodl-tmp')

from Strategy.factor.factor_base import load_all_factors
from Strategy.label.label_generator import load_label
from Strategy.model.trainer import build_panel, split_panel

# 加载
factor_dict = load_all_factors()          # 自动读取 outputs/factors/ 下所有 .fea
label_df    = load_label("TWAP_1430_1457")

# 拼接长表 Panel
panel = build_panel(factor_dict, label_df)
print(panel.shape)   # (样本数, 因子数+3)

# 按时间划分 (config.py 中已定义日期)
# Train: 2021-01-01 ~ 2023-08-01
# Val:   2023-09-01 ~ 2024-09-01
# OOS:   2024-09-01 之后
train, val, oos = split_panel(panel)

(6660324, 137)


In [6]:
from Strategy.model.trainer import XGBTrainer

trainer = XGBTrainer(
    params={
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "max_depth": 6,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 50,
        "tree_method": "hist",  # XGBoost 2+ 统一用 hist；GPU 时配合 device 见下行
        "device": "cuda",  # 无独显或报 CUDA 错时改为 "cpu" 或删此键
        "verbosity": 0,
    },
    num_boost_round=500,
    early_stopping_rounds=50,
)

trainer.train(train, val)
trainer.save_model()   # 保存到 outputs/scores/xgb_model.pkl


libgomp: Invalid value for environment variable OMP_NUM_THREADS


[0]	train-rmse:0.02893	val-rmse:0.02977
[100]	train-rmse:0.02865	val-rmse:0.02951
[200]	train-rmse:0.02853	val-rmse:0.02943
[300]	train-rmse:0.02843	val-rmse:0.02940
[400]	train-rmse:0.02835	val-rmse:0.02938
[499]	train-rmse:0.02828	val-rmse:0.02936


PosixPath('/root/autodl-tmp/Strategy/outputs/scores/xgb_model.pkl')

In [4]:
import sys
sys.path.insert(0, "/root/autodl-tmp")

from Strategy.model.trainer import XGBTrainer
from Strategy.model.scorer import generate_scores, load_scores
from Strategy.factor.factor_base import load_all_factors
from Strategy.label.label_generator import load_label
from Strategy import config

# 加载已训练好的模型
trainer = XGBTrainer().load_model(config.SCORE_OUTPUT_DIR / "xgb_model.pkl")

# 加载因子与 label。label_df 这里只用于对齐日期/股票, 不作为特征输入。
factor_dict = load_all_factors()
label_df = load_label("TWAP_1430_1457")

# 使用已训练模型重新生成全市场打分
score_path = generate_scores(
    trainer=trainer,
    factor_dict=factor_dict,
    label_df=label_df,
    model_name="xgb",
    label_tag="TWAP_1430_1457",
    normalize=True,
)

print(f"打分已保存: {score_path}")

# 后续回测直接加载该 score
score_df = load_scores("xgb", "TWAP_1430_1457")
print(score_df.shape)
score_df.tail()


libgomp: Invalid value for environment variable OMP_NUM_THREADS


打分已保存: /root/autodl-tmp/Strategy/outputs/scores/SCORE_xgb_TWAP_1430_1457.fea
(1251, 5324)


,000001,000002,000004,000005,000006,000007,000008,000009,000010,000011,...,688787,688788,688789,688793,688798,688799,688800,688819,688981,689009
TRADE_DATE,,,,,,,,,,,,,,,,,,,,,
2026-03-16,0.499015,0.500577,3.454853,NaN,0.384404,0.337840,0.685943,0.412485,0.765702,0.648303,...,-0.582531,-0.142668,0.298649,0.834107,-0.907848,0.047590,-0.767126,0.327538,-0.338470,0.530519
2026-03-17,0.298218,0.429761,2.101745,NaN,0.444932,0.034695,0.210617,0.166512,-0.367687,0.747080,...,-0.209256,-0.559455,-0.489676,0.752891,-0.527301,0.080102,0.165487,-0.195656,-0.213225,-0.125975
2026-03-18,0.214973,0.247800,-5.450637,NaN,0.382389,0.033856,0.002064,-0.039591,0.286159,0.500199,...,-0.322028,-0.217856,-0.091574,0.823234,-1.004479,0.277281,-0.548072,-0.102158,-0.673173,-0.081744
2026-03-19,0.008083,-0.065821,-6.831113,NaN,0.281984,0.083160,-0.239480,-0.287020,0.003695,0.862093,...,-1.011005,-0.340863,-0.353364,0.808876,-0.300227,0.305053,0.764021,-0.241854,-0.359580,0.099643
2026-03-20,-0.104327,0.606264,0.363937,NaN,0.038113,-0.022767,0.113482,-0.199345,0.224892,0.353317,...,-0.330513,-0.372175,-0.116382,0.875679,-0.479955,-0.413007,-0.104926,-0.325273,-0.264477,-0.479220


In [1]:
import sys
sys.path.insert(0, '/root/autodl-tmp')

from Strategy.backtest.quick_backtest import run_quick_backtest
from Strategy.model.scorer import load_scores
from Strategy.label.label_generator import load_label
from Strategy import config

score_df = load_scores("xgb", "TWAP_1430_1457")
label_df = load_label("TWAP_1430_1457")

# 可自行调整想看的 topN 曲线, 例如 (10, 20, 50, 100, 200)
TOP_KS = (20, 50, 100)

# ⚠️ 必须指定 start_date=config.VAL_START (或 OOS_START), 否则训练期的
# 样本内预测会拉高 group1 收益, 产生虚假的"高收益"假象。
quick_ret = run_quick_backtest(
    score_df=score_df,
    label_df=label_df,
    n_groups=20,
    output_dir=config.BT_RESULT_DIR,
    start_date=config.VAL_START,   # 从验证集起计, 排除训练期样本内日期
    top_ks=TOP_KS,
)

Error.  nthreads must be a positive integerOMP: Warning #80: OMP_NUM_THREADS="0": value too small.
OMP: Info #104: OMP_NUM_THREADS value "1" will be used.


In [2]:
import sys
sys.path.insert(0, '/root/autodl-tmp')

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

from Strategy.backtest.event_backtest import BacktestRunner
from Strategy.model.scorer import load_scores
from Strategy import config

score_df = load_scores("xgb", "TWAP_1430_1457")

# 可自行调整想看的每日买入 topN, 例如 (10, 20, 50, 100, 200)
TOP_NS = (20, 50, 100)
event_results = {}

for top_n in TOP_NS:
    runner = BacktestRunner(
        score_df=score_df,
        top_n=top_n,
        n_quantile_groups=20,
        rebalance_freq=1,       # 每日调仓, 即每日只买入当日 topN
        initial_capital=10_000_000.0,
        frictionless=True,      # 与 run_quick_backtest 无佣金/印花税假设对齐; 实盘改 False
    )
    result = runner.run(start_date=config.VAL_START, end_date=None)
    result.plot(save_dir=config.BT_RESULT_DIR)
    result.save_details(config.BT_RESULT_DIR)
    event_results[f"top{top_n}"] = result.nav_df.set_index("TRADE_DATE")["nav"] / result.initial_capital

# 汇总画一张 top20/top50/top100 事件回测净值对比图
nav_compare = pd.DataFrame(event_results)
fig, ax = plt.subplots(figsize=(13, 5))
for col in nav_compare.columns:
    ax.plot(nav_compare.index, nav_compare[col], label=col, lw=1.2)
ax.axvline(pd.Timestamp(config.OOS_START), color="black", ls="--", lw=0.8, alpha=0.75)
ax.axhline(1.0, color="gray", lw=0.5)
ax.set_title("Event Backtest TopN NAV Compare")
ax.set_ylabel("Net Value")
ax.legend()
plt.xticks(rotation=45, fontsize=7)
plt.tight_layout()
compare_path = config.BT_RESULT_DIR / "event_topn_nav_compare.png"
fig.savefig(compare_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"TopN 对比图已保存: {compare_path}")

TopN 对比图已保存: /root/autodl-tmp/Strategy/outputs/bt_results/event_topn_nav_compare.png


In [3]:
import sys
sys.path.insert(0, '/root/autodl-tmp')

from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from Strategy import config
from Strategy.model.scorer import load_scores
from Strategy.backtest.universe import (
    build_universe_filter,
    load_ipo_dates,
    load_out_dates,
    load_st_status,
)
from Strategy.utils.helpers import get_minute_files, date_to_int, strip_stock_prefix, is_sh_or_sz

# 新回测口径:
# - score_df.loc[T] 已经是 T-1 收盘后可得信号, T 日 09:35 open 买入。
# - quick: T 日 09:35 open 买入, T+1 日 10:00 open 卖出。
# - event: 持仓至少隔夜; 卖出日若 10:00 前涨停封死且未炸板, 则不卖出、继续持有吃连板。
# - 买入日若 09:35 open 已经涨停, 则视为买不进, 跳过。

TOP_NS_OPEN = (20, 50, 100)
BUY_TIME = 935
SELL_TIME = 1000
INITIAL_CAPITAL_OPEN = 10_000_000.0
LOT_SIZE = 100
LIMIT_TOL = 1e-4


def _standardize_wide(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "TRADE_DATE" in out.columns:
        out = out.set_index("TRADE_DATE")
    out.index = pd.DatetimeIndex(out.index).normalize()
    out.columns = pd.Index([str(c).zfill(6) for c in out.columns])
    return out.sort_index()


def _load_limit_up() -> pd.DataFrame:
    return _standardize_wide(pd.read_pickle(config.DAILY_DATA_DIR / "LIMIT_UP_PRICE.pkl"))


def _is_limit_price(price, limit_price, tol: float = LIMIT_TOL) -> bool:
    if pd.isna(price) or pd.isna(limit_price) or limit_price <= 0:
        return False
    return float(price) >= float(limit_price) * (1 - tol)


def _build_open_price_tables(start_date=None, end_date=None):
    """生成 09:35 open、10:00 open、10:00 前封涨停未炸板 三张宽表。"""
    start_int = date_to_int(start_date) if start_date is not None else None
    end_int = date_to_int(end_date) if end_date is not None else None
    files = get_minute_files(start_int, end_int)
    limit_up = _load_limit_up()

    buy_open_rows = {}
    sell_open_rows = {}
    locked_rows = {}

    for fpath in tqdm(files, desc="Build 09:35/10:00 price tables", unit="day"):
        date_int = int(fpath.stem)
        trade_date = pd.Timestamp(
            year=date_int // 10000,
            month=(date_int % 10000) // 100,
            day=date_int % 100,
        )
        df = pd.read_feather(fpath)
        df = df[df["StockID"].map(is_sh_or_sz)].copy()
        df["StockID"] = df["StockID"].map(strip_stock_prefix)

        buy_bar = df[df["time"] == BUY_TIME]
        sell_bar = df[df["time"] == SELL_TIME]
        buy_open_rows[trade_date] = buy_bar.groupby("StockID")["open"].first()
        sell_open_rows[trade_date] = sell_bar.groupby("StockID")["open"].first()

        if trade_date not in limit_up.index:
            locked_rows[trade_date] = pd.Series(dtype=bool)
            continue

        lu = limit_up.loc[trade_date]
        morning = df[(df["time"] >= 930) & (df["time"] <= SELL_TIME)].copy()
        locked = {}
        for stock, grp in morning.groupby("StockID", sort=False):
            limit_price = lu.get(stock, np.nan)
            if pd.isna(limit_price) or limit_price <= 0:
                locked[stock] = False
                continue
            grp = grp.sort_values("time")
            hit = grp[(grp["high"] >= limit_price * (1 - LIMIT_TOL)) | (grp["price"] >= limit_price * (1 - LIMIT_TOL))]
            if hit.empty:
                locked[stock] = False
                continue
            first_hit_time = hit["time"].iloc[0]
            after_hit = grp[grp["time"] >= first_hit_time]
            at_1000 = grp[grp["time"] == SELL_TIME]
            # 触板后到 10:00 所有分钟 low 都在涨停价附近, 且 10:00 仍在涨停价, 视为封死未炸板。
            locked[stock] = bool(
                not at_1000.empty
                and _is_limit_price(at_1000["open"].iloc[0], limit_price)
                and (after_hit["low"] >= limit_price * (1 - LIMIT_TOL)).all()
            )
        locked_rows[trade_date] = pd.Series(locked, dtype=bool)

    buy_open = pd.DataFrame(buy_open_rows).T.sort_index()
    sell_open = pd.DataFrame(sell_open_rows).T.sort_index()
    locked_to_1000 = pd.DataFrame(locked_rows).T.sort_index().fillna(False).astype(bool)
    for df in (buy_open, sell_open, locked_to_1000):
        df.index.name = "TRADE_DATE"
        df.columns = pd.Index([str(c).zfill(6) for c in df.columns])
    return buy_open, sell_open, locked_to_1000, limit_up


def _build_open_buyable_mask(score_df, buy_open, limit_up):
    scores = _standardize_wide(score_df)
    dates = scores.index.intersection(buy_open.index).sort_values()
    columns = scores.columns.intersection(buy_open.columns).intersection(limit_up.columns)
    scores = scores.reindex(index=dates, columns=columns)
    buy_open = buy_open.reindex(index=dates, columns=columns)
    limit_up = limit_up.reindex(index=dates, columns=columns)

    universe_ok, universe_report = build_universe_filter(
        dates=dates,
        columns=columns,
        ipo_dates=load_ipo_dates(),
        out_dates=load_out_dates(),
        st_status=load_st_status(),
        min_listing_days=20,
        delist_buffer_days=20,
        exclude_st=True,
        exclude_historical_st=True,
        excluded_prefixes=("300", "688"),
    )
    buy_limit_up = buy_open.notna() & (buy_open >= limit_up * (1 - LIMIT_TOL))
    buyable = scores.notna() & buy_open.notna() & universe_ok & ~buy_limit_up

    report = universe_report.copy()
    report["score_nonnull"] = scores.notna().sum(axis=1)
    report["has_0935_open"] = buy_open.notna().sum(axis=1)
    report["excluded_0935_limit_up"] = (scores.notna() & universe_ok & buy_limit_up).sum(axis=1)
    report["buyable"] = buyable.sum(axis=1)
    return buyable, report


def _select_top(scores: pd.Series, buyable: pd.Series, top_n: int) -> list[str]:
    s = scores.dropna()
    ok = buyable.reindex(s.index).fillna(False).astype(bool)
    return list(s.loc[ok].sort_values(ascending=False).head(top_n).index)


def run_open_quick_backtest(score_df, buy_open, sell_open, limit_up, top_ns=TOP_NS_OPEN, start_date=config.VAL_START):
    scores = _standardize_wide(score_df)
    buyable, report = _build_open_buyable_mask(scores, buy_open, limit_up)
    common_dates = scores.index.intersection(buyable.index).intersection(sell_open.index).sort_values()
    common_dates = common_dates[common_dates >= pd.Timestamp(start_date)]

    rows = {}
    diag = {}
    for i, trade_date in enumerate(common_dates[:-1]):
        sell_date = common_dates[i + 1]
        row = {}
        diag_row = {"sell_date": sell_date}
        for top_n in top_ns:
            targets = _select_top(scores.loc[trade_date], buyable.loc[trade_date], top_n)
            buy_px = buy_open.loc[trade_date].reindex(targets)
            sell_px = sell_open.loc[sell_date].reindex(targets)
            ret = sell_px / buy_px - 1
            row[f"top{top_n}"] = ret.replace([np.inf, -np.inf], np.nan).dropna().mean()
            diag_row[f"top{top_n}_selected"] = len(targets)
            diag_row[f"top{top_n}_sell_missing"] = int(sell_px.isna().sum())
        rows[trade_date] = row
        diag[trade_date] = diag_row

    ret_df = pd.DataFrame(rows).T.sort_index()
    diag_df = pd.DataFrame(diag).T.sort_index()
    ret_df.index.name = diag_df.index.name = "TRADE_DATE"
    return ret_df, diag_df, report


def _perf(ret: pd.Series, annual_days: int = 242) -> dict:
    ret = ret.dropna()
    total_ret = (1 + ret).prod() - 1 if len(ret) else np.nan
    return {
        "n_days": len(ret),
        "ann_ret_simple": ret.mean() * annual_days if len(ret) else np.nan,
        "ann_vol": ret.std() * np.sqrt(annual_days) if len(ret) else np.nan,
        "total_ret": total_ret,
        "sharpe": (ret.mean() * annual_days) / (ret.std() * np.sqrt(annual_days)) if len(ret) and ret.std() > 0 else np.nan,
    }


def plot_nav(ret_df: pd.DataFrame, title: str, save_path: Path):
    nav = (1 + ret_df).cumprod()
    fig, ax = plt.subplots(figsize=(13, 5))
    for col in nav.columns:
        p = _perf(ret_df[col])
        ax.plot(nav.index, nav[col], label=f"{col} ann={p['ann_ret_simple']:.1%}", lw=1.2)
    ax.axvline(pd.Timestamp(config.OOS_START), color="black", ls="--", lw=0.8, alpha=0.75)
    ax.axhline(1.0, color="gray", lw=0.5)
    ax.set_title(title)
    ax.set_ylabel("Net Value")
    ax.legend()
    plt.xticks(rotation=45, fontsize=7)
    plt.tight_layout()
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return nav


def run_open_event_backtest(
    score_df,
    buy_open,
    sell_open,
    locked_to_1000,
    limit_up,
    top_n: int,
    start_date=config.VAL_START,
    initial_capital: float = INITIAL_CAPITAL_OPEN,
    lot_size: int = LOT_SIZE,
):
    scores = _standardize_wide(score_df)
    buyable, _ = _build_open_buyable_mask(scores, buy_open, limit_up)
    dates = scores.index.intersection(buyable.index).intersection(sell_open.index).sort_values()
    dates = dates[dates >= pd.Timestamp(start_date)]

    cash = float(initial_capital)
    positions = {}  # stock -> {shares, entry_date, last_price}
    nav_rows, trade_rows, exception_rows = [], [], []

    for trade_date in dates:
        # 先处理已隔夜持仓的 10:00 卖出; 若 10 点前封涨停未炸板, 延后卖出并继续持有。
        for stock in list(positions.keys()):
            pos = positions[stock]
            if pd.Timestamp(pos["entry_date"]).normalize() >= trade_date:
                continue
            sell_px = sell_open.get(stock, pd.Series(dtype=float)).get(trade_date, np.nan) if stock in sell_open.columns else np.nan
            locked = bool(locked_to_1000.get(stock, pd.Series(dtype=bool)).get(trade_date, False)) if stock in locked_to_1000.columns else False
            if pd.isna(sell_px):
                exception_rows.append({"TRADE_DATE": trade_date, "StockID": stock, "reason": "10:00 open 缺失, 延后卖出"})
                continue
            if locked:
                pos["last_price"] = float(sell_px)
                exception_rows.append({"TRADE_DATE": trade_date, "StockID": stock, "reason": "10点前涨停封死未炸板, 继续持有"})
                continue
            cash += pos["shares"] * float(sell_px)
            trade_rows.append({
                "TRADE_DATE": trade_date,
                "StockID": stock,
                "side": "SELL",
                "price": float(sell_px),
                "shares": int(pos["shares"]),
                "cash_after": cash,
                "reason": "10:00 open sell",
            })
            del positions[stock]

        targets = [s for s in _select_top(scores.loc[trade_date], buyable.loc[trade_date], top_n) if s not in positions]
        budget = cash / max(len(targets), 1)
        for stock in targets:
            px = buy_open.get(stock, pd.Series(dtype=float)).get(trade_date, np.nan) if stock in buy_open.columns else np.nan
            if pd.isna(px) or px <= 0:
                exception_rows.append({"TRADE_DATE": trade_date, "StockID": stock, "reason": "09:35 open 缺失, 无法买入"})
                continue
            shares = math.floor(budget / float(px) / lot_size) * lot_size
            cost = shares * float(px)
            if shares <= 0 or cost > cash:
                continue
            cash -= cost
            positions[stock] = {"shares": shares, "entry_date": trade_date, "last_price": float(px)}
            trade_rows.append({
                "TRADE_DATE": trade_date,
                "StockID": stock,
                "side": "BUY",
                "price": float(px),
                "shares": int(shares),
                "cash_after": cash,
                "reason": "09:35 open buy",
            })

        market_value = 0.0
        for stock, pos in positions.items():
            mark = sell_open.get(stock, pd.Series(dtype=float)).get(trade_date, np.nan) if stock in sell_open.columns else np.nan
            if pd.isna(mark):
                mark = pos["last_price"]
            else:
                pos["last_price"] = float(mark)
            market_value += pos["shares"] * float(mark)
        nav_rows.append({
            "TRADE_DATE": trade_date,
            "cash": cash,
            "market_value": market_value,
            "nav": cash + market_value,
            "n_positions": len(positions),
        })

    nav_df = pd.DataFrame(nav_rows)
    trade_df = pd.DataFrame(trade_rows)
    exception_df = pd.DataFrame(exception_rows)
    return nav_df, trade_df, exception_df


score_df = load_scores("xgb", "TWAP_1430_1457")

# 需包含 start_date 后的下一交易日, 因为 quick 使用 T+1 10:00 open 卖出。
buy_open_0935, sell_open_1000, locked_to_1000, limit_up = _build_open_price_tables(start_date=config.VAL_START)

quick_open_ret, quick_open_diag, quick_open_universe_report = run_open_quick_backtest(
    score_df,
    buy_open_0935,
    sell_open_1000,
    limit_up,
    top_ns=TOP_NS_OPEN,
    start_date=config.VAL_START,
)
quick_open_nav = plot_nav(
    quick_open_ret,
    "Quick Backtest: 09:35 Open Buy, T+1 10:00 Open Sell",
    config.BT_RESULT_DIR / "open0935_1000_quick_nav.png",
)
quick_open_ret.to_csv(config.BT_RESULT_DIR / "open0935_1000_quick_returns.csv")
quick_open_diag.to_csv(config.BT_RESULT_DIR / "open0935_1000_quick_diagnostics.csv")
quick_open_universe_report.to_csv(config.BT_RESULT_DIR / "open0935_1000_universe_report.csv")
print("09:35/10:00 quick performance:")
print(pd.DataFrame({col: _perf(quick_open_ret[col]) for col in quick_open_ret.columns}).T)

open_event_navs = {}
for top_n in TOP_NS_OPEN:
    nav_df, trade_df, exception_df = run_open_event_backtest(
        score_df,
        buy_open_0935,
        sell_open_1000,
        locked_to_1000,
        limit_up,
        top_n=top_n,
        start_date=config.VAL_START,
    )
    prefix = f"open0935_1000_top{top_n}"
    nav_df.to_csv(config.BT_RESULT_DIR / f"{prefix}_event_nav.csv", index=False)
    trade_df.to_csv(config.BT_RESULT_DIR / f"{prefix}_trades_all.csv", index=False)
    exception_df.to_csv(config.BT_RESULT_DIR / f"{prefix}_exceptions.csv", index=False)
    open_event_navs[f"top{top_n}"] = nav_df.set_index("TRADE_DATE")["nav"] / INITIAL_CAPITAL_OPEN

open_event_compare = pd.DataFrame(open_event_navs)
fig, ax = plt.subplots(figsize=(13, 5))
for col in open_event_compare.columns:
    daily_ret = open_event_compare[col].pct_change().dropna()
    p = _perf(daily_ret)
    ax.plot(open_event_compare.index, open_event_compare[col], label=f"{col} ann={p['ann_ret_simple']:.1%}", lw=1.2)
ax.axvline(pd.Timestamp(config.OOS_START), color="black", ls="--", lw=0.8, alpha=0.75)
ax.axhline(1.0, color="gray", lw=0.5)
ax.set_title("Event Backtest: 09:35 Open Buy, 10:00 Open Sell Unless Locked Limit-Up")
ax.set_ylabel("Net Value")
ax.legend()
plt.xticks(rotation=45, fontsize=7)
plt.tight_layout()
compare_path = config.BT_RESULT_DIR / "open0935_1000_event_topn_nav_compare.png"
fig.savefig(compare_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"09:35/10:00 event TopN 对比图已保存: {compare_path}")

Build 09:35/10:00 price tables:   0%|          | 0/614 [00:00<?, ?day/s]

/tmp/ipykernel_133431/3223752841.py:113: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  locked_to_1000 = pd.DataFrame(locked_rows).T.sort_index().fillna(False).astype(bool)


09:35/10:00 quick performance:
        n_days  ann_ret_simple   ann_vol  total_ret    sharpe
top20    613.0       -0.061744  0.432144  -0.325485 -0.142878
top50    613.0        0.286427  0.388854   0.706312  0.736592
top100   613.0        0.406606  0.372243   1.350700  1.092313
09:35/10:00 event TopN 对比图已保存: /root/autodl-tmp/Strategy/outputs/bt_results/open0935_1000_event_topn_nav_compare.png
